# Laboratorio 7 — Spark MLlib
## 02. Estadística descriptiva, exploración y correlaciones

Se trabaja con la población analítica de **2025** (`data/processed/personas_2025.parquet`): personas de 15 años o más, ocupadas, asalariadas y con salario positivo registrado.

- Las estadísticas, los percentiles y las correlaciones se calculan con **todos** los registros de 2025 en Spark.
- A pandas solo pasan tablas agregadas (conteos, histogramas ya agrupados).
- Los resultados describen los registros analizados; no son estimaciones oficiales de la población guatemalteca (análisis no ponderado).

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, exploracion

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

config.FIGURES.mkdir(parents=True, exist_ok=True)
config.TABLES.mkdir(parents=True, exist_ok=True)

spark = config.crear_spark("lab7_02_exploracion")
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.parquet(str(config.DATA_PROCESSED / "personas_2025.parquet")).persist()
print("Registros de la población analítica 2025:", f"{df.count():,}")

## 1. Estadística descriptiva

Cantidad de observaciones, media, mediana, desviación estándar, mínimo, máximo y percentiles 25, 75 y 95 de las cuatro variables numéricas. Los percentiles son exactos, no aproximados.

In [ ]:
desc = exploracion.descriptivas(df)
desc.index = [exploracion.ETIQUETAS[v] for v in desc.index]
desc.to_csv(config.TABLES / "02_descriptivas_2025.csv")
desc

In [ ]:
d = exploracion.descriptivas(df)
s = d.loc["salario_mensual"]
print(f"Salario: mediana Q{s['mediana']:,.0f}, media Q{s['media']:,.0f}, P95 Q{s['p95']:,.0f}, máximo Q{s['maximo']:,.0f}.")
print(f"El 5 % con mayores salarios gana más de Q{s['p95']:,.0f}; el máximo es {s['maximo'] / s['mediana']:.1f} veces la mediana.")
for v in ["edad", "antiguedad", "horas_semanales"]:
    r = d.loc[v]
    print(f"{exploracion.ETIQUETAS[v]}: mediana {r['mediana']:.1f}, rango intercuartílico {r['p25']:.1f} a {r['p75']:.1f}, máximo {r['maximo']:.1f}.")

**Cómo leer la tabla.** Cuando la media queda muy por encima de la mediana y el P95 está lejos del P75, la distribución tiene una cola larga hacia valores altos. La desviación estándar mide la dispersión en las mismas unidades de la variable, por lo que solo es comparable entre variables si se mira junto con la media.

## 2. Distribución de los registros por categoría ocupacional, nivel educativo y dominio

Los códigos se muestran tal como vienen en las bases; la categoría ocupacional se rotula con su significado.

In [ ]:
CATEGORIAS = {"1": "Gobierno", "2": "Empresa privada", "3": "Jornalero o peón", "4": "Servicio doméstico"}

def rotular(tabla, mapa=None, prefijo=""):
    t = tabla.copy()
    t["etiqueta"] = [mapa.get(c, c) if mapa else (c if c == config.DESCONOCIDO else f"{prefijo}{c}") for c in t["categoria"]]
    return t

conteos = {
    "Categoría ocupacional": rotular(exploracion.conteo_categorias(df, "categoria_ocupacional"), CATEGORIAS),
    "Nivel educativo (código)": rotular(exploracion.conteo_categorias(df, "nivel_educativo")),
    "Dominio (código)": rotular(exploracion.conteo_categorias(df, "dominio")),
}

fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2))
for eje, (titulo, t) in zip(ejes, conteos.items()):
    barras = eje.bar(t["etiqueta"], t["registros"], color="#3B6FB6")
    for b, p in zip(barras, t["porcentaje"]):
        eje.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{p:.1f} %", ha="center", va="bottom", fontsize=8)
    eje.set_title(titulo)
    eje.set_ylabel("registros")
    eje.tick_params(axis="x", rotation=30)
fig.suptitle("Registros por categoría, nivel educativo y dominio (2025)", y=1.03)
plt.tight_layout()
plt.savefig(config.FIGURES / "02_distribucion_categorias.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
for titulo, t in conteos.items():
    mayor = t.loc[t["registros"].idxmax()]
    menor = t.loc[t["registros"].idxmin()]
    print(f"{titulo}: la categoría más frecuente es '{mayor['etiqueta']}' ({mayor['porcentaje']:.1f} %) y la menos frecuente '{menor['etiqueta']}' ({menor['porcentaje']:.1f} %).")
pd.concat({k: v[["etiqueta", "registros", "porcentaje"]].set_index("etiqueta") for k, v in conteos.items()})

**Interpretación.** Las barras muestran cuánto pesa cada grupo en los registros analizados. Un grupo con pocos registros produce medianas menos estables en las comparaciones de salario de más abajo, y cualquier diferencia que involucre a un grupo pequeño debe leerse con cautela. Estas proporciones describen la muestra filtrada, no a todos los trabajadores del país.

## 3. Forma de la distribución del salario

El histograma se calcula en Spark (los conteos por intervalo salen de todos los registros). Se muestra en **quetzales** y en **escala logarítmica (log10 del salario)**. La escala logarítmica solo se usa para visualizar; el objetivo del modelado sigue siendo el salario original en quetzales.

In [ ]:
h_lin = exploracion.histograma(df, "salario_mensual", bins=50)
h_log = exploracion.histograma(df, "salario_mensual", bins=50, log10=True)
sal = exploracion.asimetria(df, "salario_mensual")

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.2))
a.bar(h_lin["desde"], h_lin["registros"], width=(h_lin["hasta"] - h_lin["desde"]).iloc[0], align="edge", color="#3B6FB6")
a.axvline(sal["media"], color="#C0392B", ls="--", label=f"Media Q{sal['media']:,.0f}")
a.axvline(sal["mediana"], color="#1E8449", ls="-", label=f"Mediana Q{sal['mediana']:,.0f}")
a.set_title("Salario mensual (escala lineal, quetzales)")
a.set_xlabel("salario mensual (Q)")
a.set_ylabel("registros")
a.legend()

b.bar(h_log["desde"], h_log["registros"], width=(h_log["hasta"] - h_log["desde"]).iloc[0], align="edge", color="#3B6FB6")
b.axvline(np.log10(sal["media"]), color="#C0392B", ls="--", label="Media")
b.axvline(np.log10(sal["mediana"]), color="#1E8449", ls="-", label="Mediana")
b.set_title("Salario mensual (escala logarítmica: log10 del salario en Q)")
b.set_xlabel("log10(salario mensual en Q); 3 = Q1,000, 4 = Q10,000")
b.set_ylabel("registros")
b.legend()
plt.tight_layout()
plt.savefig(config.FIGURES / "02_histograma_salario.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
print(f"Coeficiente de asimetría (skewness) del salario: {sal['asimetria']:.2f}")
print(f"Media Q{sal['media']:,.0f} vs. mediana Q{sal['mediana']:,.0f}: la media es {100 * (sal['media'] / sal['mediana'] - 1):+.1f} % respecto a la mediana (diferencia Q{sal['media'] - sal['mediana']:,.0f}).")
if sal["asimetria"] > 0.5:
    print("Conclusión: distribución asimétrica con cola larga hacia salarios altos (asimetría positiva).")
elif sal["asimetria"] < -0.5:
    print("Conclusión: distribución asimétrica con cola hacia salarios bajos (asimetría negativa).")
else:
    print("Conclusión: distribución aproximadamente simétrica.")

**Interpretación.**

- **¿Simétrica o asimétrica?** La respuesta la da la forma del histograma y el coeficiente de asimetría impreso arriba: un valor positivo grande indica una cola derecha, es decir, pocos salarios muy altos frente a una mayoría concentrada en valores bajos o medios.
- **Media frente a mediana.** La media se ve arrastrada por los valores extremos y la mediana no. Cuando la media supera a la mediana, el salario típico de una persona (la mediana) es menor que el promedio, y usar solo el promedio sobrestimaría lo que gana la persona central. Por esa razón la mediana es la medida de resumen más informativa para el salario.
- **Escala logarítmica.** Al comprimir la cola derecha, permite ver la forma del cuerpo de la distribución; si en esa escala se acerca a una campana, la asimetría se debe sobre todo a que los salarios crecen de manera multiplicativa. El eje se lee en potencias de 10 (3 equivale a Q1,000).
- Los salarios altos y bajos **no se eliminaron**; su influencia se discutirá en el modelado.

## 4. Salario mediano por nivel educativo y por categoría ocupacional

Se usa la mediana (más robusta que la media) y se anota el número de registros de cada grupo.

In [ ]:
por_educ = rotular(exploracion.salario_por_grupo(df, "nivel_educativo"))
por_cat = rotular(exploracion.salario_por_grupo(df, "categoria_ocupacional"), CATEGORIAS)
por_dom = rotular(exploracion.salario_por_grupo(df, "dominio"))

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.4))
for eje, t, titulo in [(ejes[0], por_educ, "Nivel educativo (código)"), (ejes[1], por_cat, "Categoría ocupacional")]:
    barras = eje.bar(t["etiqueta"], t["salario_mediano"], color="#3B6FB6")
    for b, n in zip(barras, t["registros"]):
        eje.text(b.get_x() + b.get_width() / 2, b.get_height(), f"n={n:,}", ha="center", va="bottom", fontsize=8)
    eje.set_title(f"Salario mediano por {titulo.lower()}")
    eje.set_ylabel("salario mediano (Q)")
    eje.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(config.FIGURES / "02_salario_mediano_grupos.png", dpi=120, bbox_inches="tight")
plt.show()

tabla_grupos = pd.concat({
    "nivel_educativo": por_educ, "categoria_ocupacional": por_cat, "dominio": por_dom,
})[["etiqueta", "registros", "salario_mediano", "salario_medio"]]
tabla_grupos.to_csv(config.TABLES / "02_salario_por_grupo_2025.csv")
tabla_grupos

In [ ]:
def rango(t, nombre):
    alto, bajo = t.loc[t["salario_mediano"].idxmax()], t.loc[t["salario_mediano"].idxmin()]
    print(f"{nombre}: mediana más alta en '{alto['etiqueta']}' (Q{alto['salario_mediano']:,.0f}, n={alto['registros']:,}) y más baja en '{bajo['etiqueta']}' (Q{bajo['salario_mediano']:,.0f}, n={bajo['registros']:,}); razón {alto['salario_mediano'] / bajo['salario_mediano']:.1f}.")

rango(por_educ, "Nivel educativo")
rango(por_cat, "Categoría ocupacional")
rango(por_dom, "Dominio")

**Interpretación.** Si el salario mediano crece de manera sostenida al pasar de un código educativo al siguiente, la educación se asocia con mayores salarios; una ruptura del patrón puede deberse a grupos con pocos registros (fíjese en `n`). Entre categorías ocupacionales las diferencias reflejan la naturaleza del empleo (por ejemplo, el empleo de gobierno o de empresa privada frente al trabajo jornalero o el servicio doméstico). Son **asociaciones descriptivas**: no muestran que estudiar más *cause* un salario mayor ni indican cuánto debería ganar una persona, y las categorías se mezclan entre sí (quienes tienen más educación tienden a estar en ciertas categorías).

## 5. Tamaño de la muestra y salario mediano por trimestre

In [ ]:
trim = exploracion.resumen_por_trimestre(df)
trim.to_csv(config.TABLES / "02_resumen_trimestre_2025.csv", index=False)

fig, a = plt.subplots(figsize=(8, 4.2))
a.bar(trim["periodo_archivo"], trim["registros"], color="#9DB7DB", label="Registros analíticos")
a.set_ylabel("registros")
b = a.twinx()
b.plot(trim["periodo_archivo"], trim["salario_mediano"], color="#C0392B", marker="o", label="Salario mediano")
b.set_ylabel("salario mediano (Q)")
b.spines["right"].set_visible(True)
a.set_title("Muestra analítica y salario mediano por trimestre (2025)")
for x, y in zip(trim["periodo_archivo"], trim["salario_mediano"]):
    b.annotate(f"Q{y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(config.FIGURES / "02_muestra_y_mediana_trimestre.png", dpi=120, bbox_inches="tight")
plt.show()
trim

In [ ]:
variacion_n = 100 * (trim["registros"].max() / trim["registros"].min() - 1)
variacion_m = 100 * (trim["salario_mediano"].max() / trim["salario_mediano"].min() - 1)
print(f"El tamaño de la muestra varía {variacion_n:.1f} % entre el trimestre más pequeño y el más grande.")
print(f"El salario mediano varía {variacion_m:.1f} % entre el trimestre más bajo y el más alto.")

**Interpretación.** Como los cuatro archivos son cortes de una misma encuesta con rotación, se espera que el tamaño de la muestra analítica sea parecido de un trimestre a otro; una variación grande sugiere cambios en el número de ocupados asalariados con salario registrado o en la calidad del registro. Diferencias pequeñas en el salario mediano entre trimestres no permiten concluir que el salario cambió: no se ha medido incertidumbre y una parte de las personas se repite entre trimestres.

## 6. Correlaciones entre variables numéricas

Correlación de Pearson calculada con `VectorAssembler` y `Correlation.corr` de `pyspark.ml.stat`, sobre **todos** los registros elegibles de 2025.

In [ ]:
corr = exploracion.matriz_correlacion(df)
etiquetas = [exploracion.ETIQUETAS[v] for v in corr.columns]
corr_rotulada = corr.set_axis(etiquetas, axis=0).set_axis(etiquetas, axis=1)
corr_rotulada.to_csv(config.TABLES / "02_correlacion_2025.csv")
corr_rotulada.style.format("{:.3f}").background_gradient(cmap="RdBu_r", vmin=-1, vmax=1)

In [ ]:
fig, eje = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(corr_rotulada, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True,
            linewidths=.5, cbar_kws={"label": "correlación de Pearson"}, ax=eje)
eje.set_title("Matriz de correlación (2025, todos los registros)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(config.FIGURES / "02_mapa_calor_correlacion.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
con_salario = corr["salario_mensual"].drop("salario_mensual").sort_values(key=np.abs, ascending=False)
for v, r in con_salario.items():
    fuerza = "muy débil" if abs(r) < .1 else "débil" if abs(r) < .3 else "moderada" if abs(r) < .5 else "fuerte"
    print(f"Salario vs. {exploracion.ETIQUETAS[v]}: r = {r:+.3f} ({fuerza}, r² = {r ** 2:.3f})")
print(f"\nMayor asociación lineal con el salario: {exploracion.ETIQUETAS[con_salario.index[0]]}.")
r_ea = corr.loc["edad", "antiguedad"]
print(f"Edad vs. antigüedad: r = {r_ea:+.3f}")

**Interpretación.**

- **Asociación lineal con el salario.** La variable con la correlación absoluta más alta es la que mejor acompaña al salario en línea recta, pero el valor de `r²` indica qué fracción de la variación del salario se explica linealmente por esa sola variable; si es baja, ninguna variable numérica por sí sola explica bien el salario, y por eso el modelado incluye también variables categóricas (nivel educativo, categoría ocupacional y dominio).
- **Edad y antigüedad.** Es esperable una relación positiva: una persona no puede tener más años de antigüedad que de vida (regla del filtro) y, en general, quienes tienen más edad han tenido más tiempo para acumular antigüedad. Aun así, el cambio de empleo la debilita.
- **Precauciones.** Pearson mide solo asociación **lineal** y es sensible a valores extremos, y el salario tiene cola derecha; una relación no lineal o creciente por tramos puede quedar subestimada. Las correlaciones no demuestran causalidad, y las observaciones repetidas de una misma persona no son independientes.

In [ ]:
df.unpersist()
spark.stop()